# Детектор видимого фрагмента задней фигуры

Подключите данные конкурса и приватный `clevr_visible`, выберите GPU T4 ×2 и нажмите Run All. Обучение использует изображения train, оценка использует исходные сцены val.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
from IPython.display import FileLink, display
from scipy.special import expit
from sklearn.metrics import roc_auc_score

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
assert torch.cuda.device_count() == 2, "Выберите GPU T4 x2"
ROOT = Path(tempfile.mkdtemp(prefix="clevr_verify_"))
RESULT = ROOT / "verify_result"
RESULT.mkdir()
print("Рабочая папка", ROOT, flush=True)

def unpack(path, dest):
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if not (dest / name).resolve().is_relative_to(dest.resolve()):
                raise ValueError(name)
        z.extractall(dest)

def find_data(parent):
    return [p.parent for p in parent.rglob("train.csv")
            if all((p.parent / n).is_file() for n in ("val.csv","test.csv","sample_submission.csv"))
            and all((p.parent / n).is_dir() for n in ("train","val","test"))]

locations = list(INPUT.rglob("visible_manifest.json"))
if not locations:
    archives = list(INPUT.rglob("clevr_visible.zip"))
    if len(archives) != 1:
        raise ValueError("Подключите один Private Dataset из clevr_visible.zip")
    unpack(archives[0], ROOT / "package")
    locations = [ROOT / "package/visible_manifest.json"]
if len(locations) != 1:
    raise ValueError("Подключено несколько версий весов")
package = locations[0].parent
for key, digest in json.loads(locations[0].read_text()).items():
    with (package / key).open("rb") as f:
        if hashlib.file_digest(f, "sha256").hexdigest() != digest:
            raise ValueError(key)
datasets = find_data(INPUT)
if not datasets:
    for j, path in enumerate(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(path) as z:
            names = {Path(k).name for k in z.namelist()}
        if {"train.csv","val.csv","test.csv","sample_submission.csv"} <= names:
            unpack(path, ROOT / "inputs" / str(j))
    datasets = find_data(ROOT / "inputs")
if len(datasets) != 1:
    raise ValueError("Подключите данные соревнования")
DATA = datasets[0]
env = dict(os.environ, CLEVR_DATA=str(DATA), CLEVR_PRIORS=str(package/"priors"),
           CLEVR_WEIGHTS=str(package/"weights_in"), PYTHONPATH=str(package),
           PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", MKL_NUM_THREADS="2")

In [ ]:
assert all("T4" in torch.cuda.get_device_name(i) for i in range(2)), "Выберите GPU T4 ×2"
env.update(FRAGMENT_INPUT=str(package), OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", NUMEXPR_NUM_THREADS="2")
RESULT = ROOT / "fragment_detector"
RESULT.mkdir(exist_ok=True)
ARMS = ("clean", "paste", "matched")
SEEDS = (421, 853)
EPOCHS = 48
DETECTOR = "scripts.research.visible_fragment_detector"
EVAL = "scripts.research.visible_fragment_eval"

def launch(gpu, module, *args):
    step = "_".join(map(str,args)).replace("--","")
    done = RESULT / f"{step}.done"
    if done.exists():
        return
    log = RESULT / f"{step}.log"
    cmd = [sys.executable, "-m", module, *map(str,args)]
    with log.open("a", buffering=1) as out, subprocess.Popen(cmd, cwd=ROOT, text=True,
            env={**env,"CUDA_VISIBLE_DEVICES":str(gpu)}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT) as job:
        for line in job.stdout:
            print(f"[GPU {gpu}] {line}", end="", flush=True)
            out.write(line)
        if job.wait():
            shutil.copy2(log, WORK/f"visible_{step}_failed.log")
            raise RuntimeError(f"{step} завершился с ошибкой. Лог сохранён в Output.")
    done.touch()

def parallel(jobs):
    with ThreadPoolExecutor(max_workers=2) as pool:
        results = [pool.submit(fn,i,*args) for i,(fn,args) in enumerate(jobs)]
        for result in results:
            result.result()

def worker(gpu, *args):
    launch(gpu,DETECTOR,*args)

def monitor(job, stage):
    telemetry = RESULT / f"gpu_{stage}.csv"
    with telemetry.open("w",buffering=1) as out:
        query = ["nvidia-smi", "--query-gpu=timestamp,index,utilization.gpu,utilization.memory,memory.used,temperature.gpu",
                 "--format=csv", "-l", "2"]
        try:
            process = subprocess.Popen(query,stdout=out,stderr=subprocess.STDOUT)
        except FileNotFoundError:
            process = None
        try:
            return job()
        finally:
            if process is not None:
                process.terminate()
                process.wait()

## Подбор общего батча

Одновременно измеряем скорость обеих T4 на полной операции обучения с аугментацией. Учитываем выделенную и зарезервированную VRAM, время аугментации, время обучения и загрузку GPU.

In [ ]:
# matched содержит самый дорогой путь подготовки. Меряем обе T4 одновременно.
monitor(lambda: parallel([(worker,("probe","--arm","matched","--seed",seed)) for seed in SEEDS]), "probe")
probes = [json.loads((RESULT/f"probe_matched_{seed}.json").read_text()) for seed in SEEDS]
results = pd.DataFrame([dict(gpu=gpu, **r) for gpu,p in enumerate(probes) for r in p["candidates"]])
display(results)
complete = results[results.vram_fraction.le(.91)].groupby("batch").filter(lambda g:len(g)==2)
throughput = complete.groupby("batch").crops_per_s.min()
if throughput.empty:
    raise RuntimeError("Нет размера батча, который поместился на обеих T4")
BATCH = int(throughput[throughput >= .98*throughput.max()].index.max())
print("Кропов за шаг на каждой T4",BATCH,"(нативный + парный кроп на каждую сцену)")
for gpu,p in enumerate(probes):
    print("Подготовка банка CPU и загрузка на GPU",gpu,p["startup"])

## Три конфигурации по два запуска

`clean` видит исходные сцены. `paste` добавляет геометрические перекрытия из train. `matched` чаще ставит одноцветные фигуры за передними и оставляет видимыми от 4% до 36% маски. Архитектура, батч и число эпох общие.

In [ ]:
def stages():
    for arm in ARMS:
        parallel([(worker,("fit","--arm",arm,"--seed",seed,"--epochs",EPOCHS,"--batches",BATCH//2)) for seed in SEEDS])
monitor(stages,"fit")

## Исходные перекрытия и ROC-AUC

Попадание означает, что пик маски оказался в размеченном фрагменте и запрос сделан по подходящей передней фигуре. Ложные срабатывания считаются в проверенных контрольных кропах. Delta ROC-AUC измеряется поверх train-only аналога `union_tv`.

In [ ]:
def predict(split, arms):
    parallel([(worker,("prepare","--split",split,"--shard",shard)) for shard in (0,1)])
    for arm in arms:
        for seed in SEEDS:
            parallel([(worker,("predict","--arm",arm,"--seed",seed,"--split",split,"--shard",shard)) for shard in (0,1)])

monitor(lambda: predict("val",ARMS),"val")
launch(0,EVAL,"val")
report = json.loads((RESULT/"metrics.json").read_text())
cols = [dict(configuration="union_tv control",**report["baseline"])]
real = []
for arm, values in report["arms"].items():
    for seed, row in values.items():
        cols.append(dict(configuration=f"{arm} / {seed}", scenes=row["accepted_scenes"], **row["ensemble"]))
        if seed=="both":
            real.extend(dict(configuration=arm, threshold=q, **m) for q,m in row["detection"].items())
display(pd.DataFrame(cols).drop(columns="auc_by_category"))
display(pd.DataFrame(real))
print("Выбранная конфигурация",report["winner"])

## Экспорт результатов

CSV для Kaggle появится, если детектор улучшит результаты заданных проверок на val. Архив с метриками и логами сохраняется при любом исходе.

In [ ]:
winner = report["winner"]
if winner:
    monitor(lambda: predict("test",(winner,)),"test")
    launch(0,EVAL,"test")
    filename = f"rear_{winner}.csv"
    shutil.copy2(RESULT/filename,WORK/filename)
    display(FileLink(str(WORK/filename)))
with zipfile.ZipFile(WORK/"visible_results.zip","w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as out:
    for p in RESULT.iterdir():
        if p.suffix in {".csv",".json",".log",".pt",".npz"}:
            out.write(p,p.name)
display(FileLink(str(WORK/"visible_results.zip")))